# Caso 3: Aumento de peso y sedentarismo

Este cuaderno muestra un ejemplo simple de **Naive Bayes** aplicado a un problema de clasificación.

## Contexto
Un centro de salud quiere predecir el nivel de **riesgo de aumento de peso** de una persona, considerando hábitos asociados al sedentarismo, como:

- horas frente a pantalla
- nivel de actividad física
- frecuencia de comida fuera del hogar
- calidad del sueño

El objetivo es construir un modelo que clasifique si el riesgo será **Bajo**, **Medio** o **Alto**.

## 1. Importar librerías

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, ConfusionMatrixDisplay


## 2. Crear el set de datos

Cada fila representa una persona con ciertas características y el nivel de riesgo de aumento de peso observado.

In [ ]:
data = [
    [2,  'Alta',  'Baja',  'Normal', 'Bajo'],
    [3,  'Alta',  'Baja',  'Normal', 'Bajo'],
    [4,  'Media', 'Media', 'Normal', 'Medio'],
    [5,  'Media', 'Media', 'Bajo',   'Medio'],
    [6,  'Baja',  'Alta',  'Bajo',   'Alto'],
    [7,  'Baja',  'Alta',  'Bajo',   'Alto'],
    [3,  'Alta',  'Baja',  'Normal', 'Bajo'],
    [5,  'Media', 'Media', 'Normal', 'Medio'],
    [8,  'Baja',  'Alta',  'Bajo',   'Alto'],
    [6,  'Media', 'Alta',  'Bajo',   'Alto'],
    [4,  'Alta',  'Media', 'Normal', 'Bajo'],
    [5,  'Media', 'Media', 'Normal', 'Medio'],
    [7,  'Baja',  'Alta',  'Bajo',   'Alto'],
    [2,  'Alta',  'Baja',  'Normal', 'Bajo'],
    [6,  'Media', 'Alta',  'Normal', 'Medio'],
    [8,  'Baja',  'Alta',  'Bajo',   'Alto'],
    [4,  'Media', 'Media', 'Normal', 'Medio'],
    [3,  'Alta',  'Baja',  'Normal', 'Bajo']
]

df = pd.DataFrame(data, columns=['horas_pantalla', 'actividad_fisica', 'comida_fuera', 'sueno', 'riesgo_peso'])
df

## 3. Revisar el dataset

In [ ]:
print('Cantidad de registros:', df.shape[0])
print('Cantidad de columnas:', df.shape[1])
print('\nTipos de datos:')
print(df.dtypes)
print('\nDistribución de la variable objetivo:')
print(df['riesgo_peso'].value_counts())

## 4. Gráfico 1: Distribución del riesgo de aumento de peso

In [ ]:
conteo_riesgo = df['riesgo_peso'].value_counts()

plt.figure(figsize=(7,5))
plt.bar(conteo_riesgo.index, conteo_riesgo.values)
plt.title('Cantidad de personas por nivel de riesgo de aumento de peso')
plt.xlabel('Nivel de riesgo')
plt.ylabel('Cantidad')
plt.show()

## 5. Gráfico 2: Horas de pantalla promedio según nivel de riesgo

In [ ]:
promedios = df.groupby('riesgo_peso')['horas_pantalla'].mean()

plt.figure(figsize=(7,5))
plt.bar(promedios.index, promedios.values)
plt.title('Horas de pantalla promedio según nivel de riesgo')
plt.xlabel('Nivel de riesgo')
plt.ylabel('Horas promedio')
plt.show()

## 6. Codificar variables categóricas

Naive Bayes necesita trabajar con valores numéricos, por eso transformamos las columnas categóricas.

In [ ]:
encoders = {}

for col in ['actividad_fisica', 'comida_fuera', 'sueno', 'riesgo_peso']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

df

## 7. Separar variables predictoras y variable objetivo

In [ ]:
X = df.drop('riesgo_peso', axis=1)
y = df['riesgo_peso']

print('X:')
print(X.head())
print('\ny:')
print(y.head())

## 8. Dividir en entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print('Tamaño entrenamiento:', X_train.shape)
print('Tamaño prueba:', X_test.shape)
print('Distribución y_train:')
print(y_train.value_counts().sort_index())
print('\nDistribución y_test:')
print(y_test.value_counts().sort_index())

## 9. Entrenar el modelo Naive Bayes

In [ ]:
modelo = GaussianNB()
modelo.fit(X_train, y_train)

## 10. Realizar predicciones

In [ ]:
y_pred = modelo.predict(X_test)

print('Valores reales:     ', list(y_test))
print('Valores predichos:  ', list(y_pred))

## 11. Evaluar el modelo

In [ ]:
etiquetas = sorted(y.unique())
nombres_clases = encoders['riesgo_peso'].inverse_transform(etiquetas)

matriz = confusion_matrix(y_test, y_pred, labels=etiquetas)
accuracy = accuracy_score(y_test, y_pred)

print('Matriz de confusión:')
print(matriz)
print('\nAccuracy del modelo:', round(accuracy, 4))
print('\nReporte de clasificación:')
print(classification_report(
    y_test,
    y_pred,
    labels=etiquetas,
    target_names=nombres_clases,
    zero_division=0
))

## 12. Gráfico 3: Matriz de confusión

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=nombres_clases)
disp.plot()
plt.title('Matriz de confusión del modelo')
plt.show()

## 13. Probar con un nuevo caso

Caso nuevo:

- horas frente a pantalla = 7
- actividad física = Baja
- comida fuera = Alta
- sueño = Bajo

In [ ]:
nuevo_caso = pd.DataFrame([[7, 'Baja', 'Alta', 'Bajo']], columns=['horas_pantalla', 'actividad_fisica', 'comida_fuera', 'sueno'])
nuevo_caso

In [ ]:
nuevo_caso_codificado = nuevo_caso.copy()

for col in ['actividad_fisica', 'comida_fuera', 'sueno']:
    nuevo_caso_codificado[col] = encoders[col].transform(nuevo_caso_codificado[col])

nuevo_caso_codificado

In [ ]:
prediccion = modelo.predict(nuevo_caso_codificado)
resultado = encoders['riesgo_peso'].inverse_transform(prediccion)

print('Predicción del modelo:', resultado[0])

## 14. Conclusión

En este caso, Naive Bayes permite estimar el nivel de **riesgo de aumento de peso** a partir de variables simples asociadas al sedentarismo.

El cuaderno incluye gráficos para que los estudiantes puedan:

1. visualizar la distribución de clases
2. comparar las horas de pantalla promedio según riesgo
3. interpretar visualmente la matriz de confusión